# Window function implementation in PySpark

## Why are window functions needed?

### 1. Regular column operations work on one row at a time

```python
df.withColumn("tax", col("amount") * 0.18)
```

- This takes the value of `amount` from the current row and computes `tax`
- It has no awareness of other rows in the dataset

---

### 2. GroupBy + Aggregation works on groups but collapses rows

```python
df.groupBy("customer_id").agg(avg("amount").alias("avg_amount"))
```

- This groups data by `customer_id`
- Computes the average transaction amount per customer
- But reduces multiple rows into a single summary row per customer

Example:

```
Row 1  ─┐
Row 2  ─┤ → [Group] → 1 summary row (rows collapsed)
Row 3  ─┘
```

As a result, we lose the original transaction-level data.

---

### 3. The real requirement

In many real-world scenarios, we need both:
- Individual transaction rows
- Group-level statistics

Example requirement:

"For each transaction row, also show what the customer's average transaction amount is"

---

### 4. Why GroupBy alone is not sufficient

Using only `groupBy`, we would need to:
1. Aggregate the data
2. Join the result back to the original DataFrame

This approach:
- Adds unnecessary complexity
- Increases computation cost
- Is more error-prone

---

### 5. Window functions solve this problem

Window functions allow us to compute group-level statistics while preserving all original rows.

Conceptually:

```
Row 1  → Row 1 + group_stat_for_row1
Row 2  → Row 2 + group_stat_for_row2
Row 3  → Row 3 + group_stat_for_row3
```

Unlike `groupBy`, no rows are removed.

---

### 6. Core insight

| Operation  | Behavior                          |
|------------|----------------------------------|
| groupBy    | Aggregates and reduces rows      |
| Window     | Aggregates and preserves rows    |

---

Window functions are essential when we need context-aware calculations across rows without losing the original dataset structure.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col, row_number, rank, dense_rank, lag, lead, sum, avg, min, max,
    count, first, last, ntile, percent_rank, stddev, round, when, lit, abs, to_date, desc, asc
)


In [3]:
spark = SparkSession.builder.appName("BankingWindowFunctions").getOrCreate()
spark

In [4]:
df = spark.read.csv("../datasets/transactions_windows.csv", header = True, inferSchema=True)
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- credit_score: integer (nullable = true)



In [5]:
df = df.withColumn("transaction_date",
        to_date(col("transaction_date"), "yyyy-MM-dd"))

df.show(truncate=False)

+--------------+-----------+-----------+------------+------------------+----------------+--------+----------------+---------+------+------------+
|transaction_id|customer_id|name       |account_type|transaction_amount|transaction_type|balance |transaction_date|branch_id|region|credit_score|
+--------------+-----------+-----------+------------+------------------+----------------+--------+----------------+---------+------+------------+
|T001          |C101       |Arun Sharma|Savings     |5000.0            |Debit           |45000.0 |2024-01-15      |B01      |West  |720         |
|T002          |C102       |Priya Sen  |Current     |15000.0           |Credit          |120000.0|2024-01-16      |B02      |North |680         |
|T003          |C103       |Rahul Das  |Savings     |2000.0            |Debit           |8000.0  |2024-01-17      |B03      |East  |590         |
|T004          |C101       |Arun Sharma|Savings     |3000.0            |Debit           |40000.0 |2024-01-18      |B01      

### Ranking each customer's transaction by date

In [6]:
window_by_customer_date = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date")
    
df = df.withColumn("txn_sequence",
        row_number().over(window_by_customer_date))

df.select("customer_id", "name", "transaction_date",
          "transaction_amount", "txn_sequence") \
  .orderBy("customer_id", "txn_sequence") \
  .show(20)

+-----------+-----------+----------------+------------------+------------+
|customer_id|       name|transaction_date|transaction_amount|txn_sequence|
+-----------+-----------+----------------+------------------+------------+
|       C101|Arun Sharma|      2024-01-15|            5000.0|           1|
|       C101|Arun Sharma|      2024-01-18|            3000.0|           2|
|       C101|Arun Sharma|      2024-01-25|           18000.0|           3|
|       C101|Arun Sharma|      2024-01-29|            6000.0|           4|
|       C101|Arun Sharma|      2024-02-02|            4000.0|           5|
|       C102|  Priya Sen|      2024-01-16|           15000.0|           1|
|       C102|  Priya Sen|      2024-01-21|            8000.0|           2|
|       C102|  Priya Sen|      2024-01-30|           22000.0|           3|
|       C102|  Priya Sen|      2024-02-03|            5000.0|           4|
|       C103|  Rahul Das|      2024-01-17|            2000.0|           1|
|       C103|  Rahul Das|

### Get each customer's first Transaction

In [7]:
first_transactions = df \
    .withColumn("txn_seq",
        row_number().over(window_by_customer_date)) \
    .filter(col("txn_seq") == 1) \
    .drop("txn_seq")

first_transactions.select(
    "customer_id", "name",
    "transaction_date", "transaction_amount"
).show()

+-----------+-----------+----------------+------------------+
|customer_id|       name|transaction_date|transaction_amount|
+-----------+-----------+----------------+------------------+
|       C101|Arun Sharma|      2024-01-15|            5000.0|
|       C102|  Priya Sen|      2024-01-16|           15000.0|
|       C103|  Rahul Das|      2024-01-17|            2000.0|
|       C104| Meena Iyer|      2024-01-19|           25000.0|
|       C105| Suresh Roy|      2024-01-20|            1000.0|
|       C106|Fatima Khan|      2024-01-22|            4500.0|
|       C107|Vikram Nair|      2024-01-24|            7000.0|
+-----------+-----------+----------------+------------------+



### Get customer's most recent transaction

In [10]:
window_desc = Window \
    .partitionBy("customer_id") \
    .orderBy(desc("transaction_date"))

latest_transactions = df \
    .withColumn("txn_seq",
        row_number().over(window_desc)) \
    .filter(col("txn_seq") == 1) \
    .drop("txn_seq")

latest_transactions.show()

+--------------+-----------+-----------+------------+------------------+----------------+--------+----------------+---------+------+------------+------------+
|transaction_id|customer_id|       name|account_type|transaction_amount|transaction_type| balance|transaction_date|branch_id|region|credit_score|txn_sequence|
+--------------+-----------+-----------+------------+------------------+----------------+--------+----------------+---------+------+------------+------------+
|          T019|       C101|Arun Sharma|     Savings|            4000.0|           Debit| 18000.0|      2024-02-02|      B01|  West|         720|           5|
|          T020|       C102|  Priya Sen|     Current|            5000.0|           Debit|122000.0|      2024-02-03|      B02| North|         680|           4|
|          T017|       C103|  Rahul Das|     Savings|             500.0|           Debit|  7500.0|      2024-01-31|      B03|  East|         590|           3|
|          T018|       C104| Meena Iyer|      

### Rank branches by total transaction volume within each region

In [11]:
branch_volumes = df.groupBy("branch_id", "region").agg(
    round(sum("transaction_amount"), 2).alias("total_volume")
)

window_by_region = Window \
    .partitionBy("region") \
    .orderBy(desc("total_volume"))

branch_volumes = branch_volumes \
    .withColumn("rank_in_region",
        rank().over(window_by_region)) \
    .withColumn("dense_rank_in_region",
        dense_rank().over(window_by_region)) \
    .withColumn("row_num_in_region",
        row_number().over(window_by_region))

branch_volumes.orderBy("region", "rank_in_region").show()

+---------+------+------------+--------------+--------------------+-----------------+
|branch_id|region|total_volume|rank_in_region|dense_rank_in_region|row_num_in_region|
+---------+------+------------+--------------+--------------------+-----------------+
|      B03|  East|     14500.0|             1|                   1|                1|
|      B02| North|     50000.0|             1|                   1|                1|
|      B04| South|     33000.0|             1|                   1|                1|
|      B07| South|     16000.0|             2|                   2|                2|
|      B06| South|      7500.0|             3|                   3|                3|
|      B01|  West|     36000.0|             1|                   1|                1|
|      B05|  West|      3000.0|             2|                   2|                2|
+---------+------+------------+--------------+--------------------+-----------------+



### Dividing data into N equal buckets

In [13]:
window_all = Window.orderBy(desc("transaction_amount"))

df = df.withColumn("amount_quartile",
        ntile(4).over(window_all))

df.select("customer_id", "transaction_amount", "amount_quartile") \
  .orderBy("amount_quartile", desc("transaction_amount")) \
  .show(20)

+-----------+------------------+---------------+
|customer_id|transaction_amount|amount_quartile|
+-----------+------------------+---------------+
|       C104|           25000.0|              1|
|       C102|           22000.0|              1|
|       C101|           18000.0|              1|
|       C102|           15000.0|              1|
|       C103|           12000.0|              1|
|       C107|            9000.0|              2|
|       C102|            8000.0|              2|
|       C104|            8000.0|              2|
|       C107|            7000.0|              2|
|       C101|            6000.0|              2|
|       C101|            5000.0|              3|
|       C102|            5000.0|              3|
|       C106|            4500.0|              3|
|       C101|            4000.0|              3|
|       C101|            3000.0|              3|
|       C106|            3000.0|              4|
|       C103|            2000.0|              4|
|       C105|       

In [15]:
# Decile analysis — divide into 10 buckets by credit score
window_credit = Window.orderBy("credit_score")

df = df.withColumn("credit_decile",
        ntile(10).over(window_credit))

df.select("customer_id", "transaction_amount", "credit_score",'credit_decile') \
  .orderBy("credit_decile", desc("credit_score")) \
  .show(20)

+-----------+------------------+------------+-------------+
|customer_id|transaction_amount|credit_score|credit_decile|
+-----------+------------------+------------+-------------+
|       C103|            2000.0|         590|            1|
|       C103|           12000.0|         590|            1|
|       C104|           25000.0|         610|            2|
|       C103|             500.0|         590|            2|
|       C105|            1000.0|         640|            3|
|       C104|            8000.0|         610|            3|
|       C102|           15000.0|         680|            4|
|       C105|            2000.0|         640|            4|
|       C102|            8000.0|         680|            5|
|       C102|           22000.0|         680|            5|
|       C106|            4500.0|         710|            6|
|       C102|            5000.0|         680|            6|
|       C101|            5000.0|         720|            7|
|       C106|            3000.0|        

### Percentile position

In [17]:
# What percentile is each transaction amount within its account type?

window_acct = Window \
    .partitionBy("account_type") \
    .orderBy("transaction_amount")

df = df.withColumn("amount_percentile",
        round(percent_rank().over(window_acct) * 100, 1))

df.select("customer_id", "account_type",
          "transaction_amount", "amount_percentile") \
  .orderBy("account_type", "amount_percentile") \
  .show()

# A value of 80.0 means this transaction amount is higher than 80% of transactions in the same account type

+-----------+------------+------------------+-----------------+
|customer_id|account_type|transaction_amount|amount_percentile|
+-----------+------------+------------------+-----------------+
|       C106|     Current|            3000.0|              0.0|
|       C106|     Current|            4500.0|             20.0|
|       C102|     Current|            5000.0|             40.0|
|       C102|     Current|            8000.0|             60.0|
|       C102|     Current|           15000.0|             80.0|
|       C102|     Current|           22000.0|            100.0|
|       C104|        Loan|            8000.0|              0.0|
|       C104|        Loan|           25000.0|            100.0|
|       C103|     Savings|             500.0|              0.0|
|       C105|     Savings|            1000.0|              9.1|
|       C103|     Savings|            2000.0|             18.2|
|       C105|     Savings|            2000.0|             18.2|
|       C101|     Savings|            30

### lag and lead function:
- lag() looks backwards
- lead() looks forward

In [19]:
# For each transaction, show the PREVIOUS transaction amount
# Used to detect sudden spikes

window_cust_date = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date")

df = df.withColumn("prev_txn_amount",
        lag(col("transaction_amount"), 1).over(window_cust_date))


df.select(
    "customer_id", "transaction_date",
    "transaction_amount", "prev_txn_amount"
).orderBy("customer_id", "transaction_date").show()

+-----------+----------------+------------------+---------------+
|customer_id|transaction_date|transaction_amount|prev_txn_amount|
+-----------+----------------+------------------+---------------+
|       C101|      2024-01-15|            5000.0|           NULL|
|       C101|      2024-01-18|            3000.0|         5000.0|
|       C101|      2024-01-25|           18000.0|         3000.0|
|       C101|      2024-01-29|            6000.0|        18000.0|
|       C101|      2024-02-02|            4000.0|         6000.0|
|       C102|      2024-01-16|           15000.0|           NULL|
|       C102|      2024-01-21|            8000.0|        15000.0|
|       C102|      2024-01-30|           22000.0|         8000.0|
|       C102|      2024-02-03|            5000.0|        22000.0|
|       C103|      2024-01-17|            2000.0|           NULL|
|       C103|      2024-01-23|           12000.0|         2000.0|
|       C103|      2024-01-31|             500.0|        12000.0|
|       C1

In [20]:
# Computing the change from the prev transaction
df = df.withColumn("amount_change",
        col("transaction_amount") - col("prev_txn_amount"))

df = df.withColumn("pct_change",
        round(
            (col("transaction_amount") - col("prev_txn_amount"))
            / col("prev_txn_amount") * 100,
        1))

# FRAUD DETECTION: flagging if current transaction is 3x more than previous
df = df.withColumn("sudden_spike_flag",
    when(
        col("transaction_amount") > col("prev_txn_amount") * 3,
        "SPIKE ALERT"
    ).otherwise("Normal")
)

df.filter(col("sudden_spike_flag") == "SPIKE ALERT") \
  .select("customer_id", "name", "transaction_date",
          "prev_txn_amount", "transaction_amount",
          "pct_change", "sudden_spike_flag") \
  .show(truncate=False)

+-----------+-----------+----------------+---------------+------------------+----------+-----------------+
|customer_id|name       |transaction_date|prev_txn_amount|transaction_amount|pct_change|sudden_spike_flag|
+-----------+-----------+----------------+---------------+------------------+----------+-----------------+
|C101       |Arun Sharma|2024-01-25      |3000.0         |18000.0           |500.0     |SPIKE ALERT      |
|C103       |Rahul Das  |2024-01-23      |2000.0         |12000.0           |500.0     |SPIKE ALERT      |
+-----------+-----------+----------------+---------------+------------------+----------+-----------------+



### For each transaction, show the next transaction date, used to compute "days until next activity"

In [22]:
df = df.withColumn("next_txn_date",
        lead(col("transaction_date"), 1).over(window_cust_date))

df = df.withColumn("days_to_next_txn",
        when(col("next_txn_date").isNotNull(),
             col("next_txn_date") -
             col("transaction_date"))
        .otherwise(lit(None)))

df.select("customer_id", "transaction_date",
          "next_txn_date", "days_to_next_txn") \
  .orderBy("customer_id", "transaction_date") \
  .show()

+-----------+----------------+-------------+-----------------+
|customer_id|transaction_date|next_txn_date| days_to_next_txn|
+-----------+----------------+-------------+-----------------+
|       C101|      2024-01-15|   2024-01-18| INTERVAL '3' DAY|
|       C101|      2024-01-18|   2024-01-25| INTERVAL '7' DAY|
|       C101|      2024-01-25|   2024-01-29| INTERVAL '4' DAY|
|       C101|      2024-01-29|   2024-02-02| INTERVAL '4' DAY|
|       C101|      2024-02-02|         NULL|             NULL|
|       C102|      2024-01-16|   2024-01-21| INTERVAL '5' DAY|
|       C102|      2024-01-21|   2024-01-30| INTERVAL '9' DAY|
|       C102|      2024-01-30|   2024-02-03| INTERVAL '4' DAY|
|       C102|      2024-02-03|         NULL|             NULL|
|       C103|      2024-01-17|   2024-01-23| INTERVAL '6' DAY|
|       C103|      2024-01-23|   2024-01-31| INTERVAL '8' DAY|
|       C103|      2024-01-31|         NULL|             NULL|
|       C104|      2024-01-19|   2024-02-01|INTERVAL '1

### Cumulative money flow per customer with time 

In [24]:
window_running = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)


df = df.withColumn("cumulative_txn_amount",
        sum(col("transaction_amount")).over(window_running))

df.select(
    "customer_id", "name",
    "transaction_date", "transaction_amount",
    "cumulative_txn_amount"
).orderBy("customer_id", "transaction_date").show(20)

+-----------+-----------+----------------+------------------+---------------------+
|customer_id|       name|transaction_date|transaction_amount|cumulative_txn_amount|
+-----------+-----------+----------------+------------------+---------------------+
|       C101|Arun Sharma|      2024-01-15|            5000.0|               5000.0|
|       C101|Arun Sharma|      2024-01-18|            3000.0|               8000.0|
|       C101|Arun Sharma|      2024-01-25|           18000.0|              26000.0|
|       C101|Arun Sharma|      2024-01-29|            6000.0|              32000.0|
|       C101|Arun Sharma|      2024-02-02|            4000.0|              36000.0|
|       C102|  Priya Sen|      2024-01-16|           15000.0|              15000.0|
|       C102|  Priya Sen|      2024-01-21|            8000.0|              23000.0|
|       C102|  Priya Sen|      2024-01-30|           22000.0|              45000.0|
|       C102|  Priya Sen|      2024-02-03|            5000.0|              5

### Running average transaction amount per customer

In [26]:
df = df.withColumn("running_avg_amount",
        round(
            avg(col("transaction_amount")).over(window_running),
            2
        ))

df = df.withColumn("running_std",
        stddev(col("transaction_amount")).over(window_running))

df = df.withColumn("anomaly_flag",
    when(
        col("transaction_amount") >
        col("running_avg_amount") + (2 * col("running_std")),
        "ANOMALY"
    ).otherwise("Normal")
)

df.select(
    "customer_id", "transaction_date",
    "transaction_amount", "running_avg_amount",
    "running_std", "anomaly_flag"
).orderBy("customer_id", "transaction_date").show(20, truncate=False)

+-----------+----------------+------------------+------------------+------------------+------------+
|customer_id|transaction_date|transaction_amount|running_avg_amount|running_std       |anomaly_flag|
+-----------+----------------+------------------+------------------+------------------+------------+
|C101       |2024-01-15      |5000.0            |5000.0            |NULL              |Normal      |
|C101       |2024-01-18      |3000.0            |4000.0            |1414.213562373095 |Normal      |
|C101       |2024-01-25      |18000.0           |8666.67           |8144.527815247077 |Normal      |
|C101       |2024-01-29      |6000.0            |8000.0            |6782.329983125268 |Normal      |
|C101       |2024-02-02      |4000.0            |7200.0            |6140.032573203501 |Normal      |
|C102       |2024-01-16      |15000.0           |15000.0           |NULL              |Normal      |
|C102       |2024-01-21      |8000.0            |11500.0           |4949.747468305833 |Norm

### Rolling window aggregation - helps to smoothen the spikes and find trend

In [28]:
window_rolling_3 = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date") \
    .rowsBetween(-2, Window.currentRow)


df = df.withColumn("rolling_3txn_avg",
        round(
            avg("transaction_amount").over(window_rolling_3),
            2
        ))

df.select(
    "customer_id", "transaction_date",
    "transaction_amount", "rolling_3txn_avg"
).orderBy("customer_id", "transaction_date").show(20)

+-----------+----------------+------------------+----------------+
|customer_id|transaction_date|transaction_amount|rolling_3txn_avg|
+-----------+----------------+------------------+----------------+
|       C101|      2024-01-15|            5000.0|          5000.0|
|       C101|      2024-01-18|            3000.0|          4000.0|
|       C101|      2024-01-25|           18000.0|         8666.67|
|       C101|      2024-01-29|            6000.0|          9000.0|
|       C101|      2024-02-02|            4000.0|         9333.33|
|       C102|      2024-01-16|           15000.0|         15000.0|
|       C102|      2024-01-21|            8000.0|         11500.0|
|       C102|      2024-01-30|           22000.0|         15000.0|
|       C102|      2024-02-03|            5000.0|        11666.67|
|       C103|      2024-01-17|            2000.0|          2000.0|
|       C103|      2024-01-23|           12000.0|          7000.0|
|       C103|      2024-01-31|             500.0|         4833

### First and last transaction amount for a customer 

In [30]:
window_full = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date") \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# First transaction amount ever for each customer
df = df.withColumn("first_txn_amount",
        first("transaction_amount").over(window_full))

# Last transaction amount ever for each customer
df = df.withColumn("last_txn_amount",
        last("transaction_amount").over(window_full))

# Computing change from first to last transaction
df = df.withColumn("total_behaviour_change",
        col("last_txn_amount") - col("first_txn_amount"))

df.select(
    "customer_id", "transaction_date",
    "transaction_amount",
    "first_txn_amount", "last_txn_amount",
    "total_behaviour_change"
).orderBy("customer_id", "transaction_date").show()

+-----------+----------------+------------------+----------------+---------------+----------------------+
|customer_id|transaction_date|transaction_amount|first_txn_amount|last_txn_amount|total_behaviour_change|
+-----------+----------------+------------------+----------------+---------------+----------------------+
|       C101|      2024-01-15|            5000.0|          5000.0|         4000.0|               -1000.0|
|       C101|      2024-01-18|            3000.0|          5000.0|         4000.0|               -1000.0|
|       C101|      2024-01-25|           18000.0|          5000.0|         4000.0|               -1000.0|
|       C101|      2024-01-29|            6000.0|          5000.0|         4000.0|               -1000.0|
|       C101|      2024-02-02|            4000.0|          5000.0|         4000.0|               -1000.0|
|       C102|      2024-01-16|           15000.0|         15000.0|         5000.0|              -10000.0|
|       C102|      2024-01-21|            8000